In [ ]:
!pip install -q ultralytics

import os, glob, random, shutil, time, yaml
import numpy as np
import pandas as pd
import cv2
from ultralytics import YOLO


In [ ]:
from google.colab import drive
import zipfile

drive.mount('/content/drive')
zip_path = '/content/drive/MyDrive/Colab Notebooks/dataset/dataset_kontainer_colab.zip'
root = '/content/dataset_kontainer'

if not os.path.exists(root):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(root)

train_all = sorted(glob.glob(f'{root}/train/images/*'))
bg_imgs = [f for f in train_all if 'bg_' in os.path.basename(f)]
dmg_train = [f for f in train_all if 'bg_' not in os.path.basename(f)]


In [ ]:
scenarios = {
    'S1': '/content/data_s1',
    'S2': '/content/data_s2',
    'S3': '/content/data_s3'
}

def sync_split(src_list, dst_dir):
    os.makedirs(f'{dst_dir}/train/images', exist_ok=True)
    os.makedirs(f'{dst_dir}/train/labels', exist_ok=True)
    for p in src_list:
        shutil.copy(p, f'{dst_dir}/train/images/')
        lbl = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
        if os.path.exists(lbl):
            shutil.copy(lbl, f'{dst_dir}/train/labels/')
        else:
            open(f'{dst_dir}/train/labels/{os.path.basename(lbl)}', 'w').close()
    for s in ['val', 'test']:
        shutil.copytree(f'{root}/{s}', f'{dst_dir}/{s}', dirs_exist_ok=True)

# S1: 571 damaged
sync_split(dmg_train, scenarios['S1'])

# S2: 571 damaged + 217 offline augmented (isovolume 788)
sync_split(dmg_train, scenarios['S2'])
random.seed(42)
for idx, p in enumerate(random.sample(dmg_train, 217)):
    img = cv2.imread(p)
    lbl = p.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    boxes = [line.strip().split() for line in open(lbl)] if os.path.exists(lbl) else []
    
    out_img = f"{scenarios['S2']}/train/images/aug_{idx}_{os.path.basename(p)}"
    out_lbl = f"{scenarios['S2']}/train/labels/aug_{idx}_{os.path.basename(lbl)}"
    
    cv2.imwrite(out_img, cv2.flip(img, 1))
    with open(out_lbl, 'w') as f:
        for b in boxes:
            f.write(f"{b[0]} {1.0 - float(b[1]):.6f} {b[2]} {b[3]} {b[4]}\n")

# S3 & S4: 571 damaged + 217 background (total 788)
sync_split(train_all, scenarios['S3'])

for k, d in scenarios.items():
    cfg = {'path': d, 'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
           'nc': 4, 'names': ['Dent', 'Rust', 'Hole', 'Deframe']}
    with open(f'/content/{k.lower()}.yaml', 'w') as f:
        yaml.dump(cfg, f)


In [ ]:
runs = [
    ('S1', 'yolov8n.pt', '/content/s1.yaml'),
    ('S2', 'yolov8n.pt', '/content/s2.yaml'),
    ('S3', 'yolov8n.pt', '/content/s3.yaml'),
    ('S4', 'yolov8s.pt', '/content/s3.yaml')
]

# Mode cepat: 1 seed terkunci (seed=42), 30 epoch, patience 8
for sc, model_id, yml in runs:
    name = f'{sc}_s42'
    if os.path.exists(f'/content/runs/{name}/weights/best.pt'):
        print(f'{name} sudah selesai, lewati.')
        continue
    print(f'Melatih {name} ({model_id})...')
    model = YOLO(model_id)
    model.train(
        data=yml, epochs=30, patience=8, imgsz=640, batch=16,
        optimizer='SGD', lr0=0.01, seed=42, project='/content/runs',
        name=name, plots=False, verbose=False
    )


In [ ]:
records = []
for sc, model_id, yml in runs:
    pt = f'/content/runs/{sc}_s42/weights/best.pt'
    if not os.path.exists(pt):
        continue
    m = YOLO(pt).val(data=yml, split='test', imgsz=640, conf=0.001, verbose=False)
    records.append({
        'scenario': sc, 'model': model_id.replace('.pt', ''),
        'precision': round(float(m.box.mp), 4),
        'recall': round(float(m.box.mr), 4),
        'mAP50': round(float(m.box.map50), 4),
        'mAP50-95': round(float(m.box.map), 4),
        'dent': round(float(m.box.ap50[0]), 4) if len(m.box.ap50) > 0 else 0,
        'rust': round(float(m.box.ap50[1]), 4) if len(m.box.ap50) > 1 else 0,
        'hole': round(float(m.box.ap50[2]), 4) if len(m.box.ap50) > 2 else 0,
        'deframe': round(float(m.box.ap50[3]), 4) if len(m.box.ap50) > 3 else 0,
    })

df = pd.DataFrame(records)
print(df.to_string(index=False))
df.to_csv('/content/hasil_komparasi_test.csv', index=False)


In [ ]:
test_imgs = glob.glob(f'{root}/test/images/*')[:50]
for sc in ['S3', 'S4']:
    pt = f'/content/runs/{sc}_s42/weights/best.pt'
    if not os.path.exists(pt):
        continue
    m = YOLO(pt)
    for dev in ['cuda', 'cpu']:
        for p in test_imgs[:5]:
            m(p, device=dev, verbose=False)
        t0 = time.perf_counter()
        for p in test_imgs:
            m(p, device=dev, verbose=False)
        lat = (time.perf_counter() - t0) / len(test_imgs) * 1000
        print(f'{sc} ({dev}): {lat:.2f} ms ({1000/lat:.1f} FPS)')
